# Fraud Detection Model Training

This notebook trains a Logistic Regression model for fraud detection based on specific features required by the decision engine:
1. `Transaction_Amount`
2. `user_transaction_count_24h`
3. `location_mismatch`

It loads local training and testing data, performs feature engineering, trains the model, evaluates it, and saves the model (`model.onnx`) and scaler (`scaler.joblib`).

## 1. Imports

In [13]:
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d kartik2112/fraud-detection

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0


In [14]:
!unzip fraud-detection.zip

Archive:  fraud-detection.zip
  inflating: fraudTest.csv           
  inflating: fraudTrain.csv          


In [2]:
!pip install skl2onnx onnxruntime

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import math
import joblib  # For saving the scaler

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# ONNX imports
import skl2onnx
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

import warnings

warnings.filterwarnings("ignore")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.8 MB/s eta 0:00:00


## 2. Load Data

Load the training and testing datasets locally.

In [3]:
try:
    train_df = pd.read_csv("fraudTrain.csv")
    test_df = pd.read_csv("fraudTest.csv")
    print("Datasets loaded successfully.")
    print("Train shape:", train_df.shape)
    print("Test shape:", test_df.shape)
except FileNotFoundError as e:
    print(f"Error loading data: {e}")
    print("Please ensure 'fraudTrain.csv' and 'fraudTest.csv' exists.")
    raise e

print("\nTraining Data Head:")
print(train_df.head())

Datasets loaded successfully.
Train shape: (1296675, 23)
Test shape: (555719, 23)

Training Data Head:
   Unnamed: 0 trans_date_trans_time            cc_num  \
0           0   2019-01-01 00:00:18  2703186189652095   
1           1   2019-01-01 00:00:44      630423337322   
2           2   2019-01-01 00:00:51    38859492057661   
3           3   2019-01-01 00:01:16  3534093764340240   
4           4   2019-01-01 00:03:06   375534208663984   

                             merchant       category     amt      first  \
0          fraud_Rippin, Kub and Mann       misc_net    4.97   Jennifer   
1     fraud_Heller, Gutmann and Zieme    grocery_pos  107.23  Stephanie   
2                fraud_Lind-Buckridge  entertainment  220.11     Edward   
3  fraud_Kutch, Hermiston and Farrell  gas_transport   45.00     Jeremy   
4                 fraud_Keeling-Crist       misc_pos   41.96      Tyler   

      last gender                        street  ...      lat      long  \
0    Banks      F           

## 3. Feature Engineering

Create the three required features: `Transaction_Amount`, `location_mismatch`, and `user_transaction_count_24h`.

### 3.1 Haversine Distance Function

Function to calculate distance between two lat/lon points.

In [4]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance in kilometers between two points
    on the earth (specified in decimal degrees)
    """
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    c = 2 * math.asin(math.sqrt(a))
    r = 6371  # Radius of earth in kilometers. Use 3956 for miles. Determines return value units.
    return c * r

### 3.2 Feature Creation Function

Function to apply feature engineering steps to a dataframe.

In [5]:
def create_features(df):
    # Convert to datetime
    df["trans_datetime"] = pd.to_datetime(df["trans_date_trans_time"])

    # 1. Transaction_Amount
    df["Transaction_Amount"] = df["amt"]

    # 2. location_mismatch
    # Calculate distance and apply threshold (e.g., 100km)
    distances = []
    for index, row in df.iterrows():
        try:
            dist = haversine(
                row["lat"], row["long"], row["merch_lat"], row["merch_long"]
            )
            distances.append(dist)
        except (
            ValueError
        ):  # Handle potential math domain errors if coordinates are invalid
            distances.append(np.nan)

    df["distance_km"] = distances
    distance_threshold = 100  # Define the threshold in KM
    df["location_mismatch"] = (df["distance_km"] > distance_threshold).astype(int)

    # 3. user_transaction_count_24h
    # Ensure data is sorted by user and time for efficient calculation
    df = df.sort_values(by=["cc_num", "trans_datetime"])

    # Use unix time for easier comparison
    df["unix_time"] = df["trans_datetime"].astype(np.int64) // 10**9

    # Calculate counts efficiently using groupby and apply with searchsorted
    def count_past_24h(group):
        timestamps = group["unix_time"].values
        counts = np.zeros(len(group), dtype=int)
        for i in range(len(group)):
            current_time = timestamps[i]
            cutoff_time = current_time - 24 * 3600  # 24 hours in seconds
            # Find the index of the first transaction within the 24h window
            # We look for transactions >= cutoff_time and < current_time
            start_index = np.searchsorted(timestamps[:i], cutoff_time, side="left")
            counts[i] = (
                i - start_index
            )  # Count transactions from start_index up to (but not including) i
        group["user_transaction_count_24h"] = counts
        return group

    print("Calculating user_transaction_count_24h (this may take a while)...")
    df = df.groupby("cc_num").apply(count_past_24h)
    print("Calculation complete.")

    # Select final features and target
    required_features = [
        "Transaction_Amount",
        "location_mismatch",
        "user_transaction_count_24h",
    ]
    target = "is_fraud"

    # Handle potential NaNs created during feature engineering (e.g., distance calc)
    df = df.dropna(subset=required_features + [target])

    return df, required_features, target


# Apply feature engineering
train_df_featured, features, target = create_features(
    train_df.copy()
)  # Use copy to avoid modifying original df
test_df_featured, _, _ = create_features(test_df.copy())

print("\nFeature Engineering Complete.")
print("Train featured shape:", train_df_featured.shape)
print("Test featured shape:", test_df_featured.shape)
print("Features:", features)
print(train_df_featured[features + [target]].head())

Calculating user_transaction_count_24h (this may take a while)...
Calculation complete.
Calculating user_transaction_count_24h (this may take a while)...
Calculation complete.

Feature Engineering Complete.
Train featured shape: (1296675, 28)
Test featured shape: (555719, 28)
Features: ['Transaction_Amount', 'location_mismatch', 'user_transaction_count_24h']
                  Transaction_Amount  location_mismatch  \
cc_num                                                    
60416207185 1017                7.27                  1   
            2724               52.94                  1   
            2726               82.08                  0   
            2882               34.79                  0   
            2907               27.18                  0   

                  user_transaction_count_24h  is_fraud  
cc_num                                                  
60416207185 1017                           0         0  
            2724                           1         0

## 4. Data Splitting and Scaling

In [6]:
# Prepare data for scikit-learn
X_train = train_df_featured[features]
y_train = train_df_featured[target]
X_test = test_df_featured[features]
y_test = test_df_featured[target]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled.")

# Save the scaler
scaler_filename = "scaler.joblib"
joblib.dump(scaler, scaler_filename)
print(f"Scaler saved to {scaler_filename}")

Data scaled.
Scaler saved to scaler.joblib


## 5. Model Training (Logistic Regression)

In [7]:
# Initialize and train the model
model = LogisticRegression(solver="liblinear", random_state=42, class_weight="balanced")
model.fit(X_train_scaled, y_train)

print("Model trained.")

Model trained.


## 6. Model Evaluation

In [8]:
# Predict on the test set
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[
    :, 1
]  # Probability of class 1 (fraud)

# Print evaluation metrics
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    print(f"\nROC AUC Score: {roc_auc:.4f}")
except ValueError as e:
    print(f"\nCould not calculate ROC AUC Score: {e}")

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    553574
           1       0.05      0.75      0.10      2145

    accuracy                           0.95    555719
   macro avg       0.53      0.85      0.54    555719
weighted avg       1.00      0.95      0.97    555719

Confusion Matrix:
[[525199  28375]
 [   532   1613]]

ROC AUC Score: 0.8259


## 7. Export Model to ONNX

Save the trained model in ONNX format for use by the decision engine.

In [9]:
# Define the input type for the ONNX model
# It expects a float tensor with shape [batch_size, num_features]
# We have 3 features: Transaction_Amount, location_mismatch, user_transaction_count_24h
initial_type = [
    ("float_input", FloatTensorType([None, X_train_scaled.shape[1]]))
]  # Use shape[1] for number of features

# Convert the model to ONNX format
onnx_model = skl2onnx.convert_sklearn(model, initial_types=initial_type)

# Save the ONNX model
model_filename = "model.onnx"
with open(model_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model saved to {model_filename}")

Model saved to model.onnx


## 8. Verify ONNX Model

Load the saved ONNX model and make a prediction to ensure it works.

In [10]:
try:
    sess = ort.InferenceSession(model_filename)
    input_name = sess.get_inputs()[0].name
    output_names = [output.name for output in sess.get_outputs()]
    print(f"ONNX Model Input: {input_name}")
    print(f"ONNX Model Outputs: {output_names}")

    # Prepare a sample input (first row of scaled test data)
    sample_input = X_test_scaled[0:1].astype(np.float32)  # Ensure type is float32

    # Run inference
    # The output order might be [label, probabilities]
    results = sess.run(output_names, {input_name: sample_input})

    print("\nONNX Model Prediction (Sample):")
    print(f"Predicted Label: {results[0][0]}")
    # The second output usually contains probabilities for each class
    if len(results) > 1:
        print(f"Predicted Probabilities: {results[1][0]}")
        # Extract probability of fraud (class 1)
        fraud_prob = (
            results[1][0].get(1, "N/A")
            if isinstance(results[1][0], dict)
            else results[1][0][1]
        )
        print(f"Predicted Fraud Probability (Class 1): {fraud_prob}")

    print("\nONNX model verified successfully.")

except Exception as e:
    print(f"\nError verifying ONNX model: {e}")

ONNX Model Input: float_input
ONNX Model Outputs: ['output_label', 'output_probability']

ONNX Model Prediction (Sample):
Predicted Label: 0
Predicted Probabilities: {0: 0.6721222996711731, 1: 0.3278777003288269}
Predicted Fraud Probability (Class 1): 0.3278777003288269

ONNX model verified successfully.
